## 1. Setup

Install/import dependencies and load environment variables from `.env`.

In [ ]:
import os, json, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath("../.."))
from dotenv import load_dotenv
from agent_memory_toolkit.aio import AsyncCosmosMemoryClient

# Load environment variables from .env in the repo root

load_dotenv(os.path.join("../..", ".env"))
print("COSMOS_DB_ENDPOINT:", os.getenv("COSMOS_DB_ENDPOINT"))
print("COSMOS_DB_DATABASE:", os.getenv("COSMOS_DB_DATABASE"))
print("COSMOS_DB_CONTAINER:", os.getenv("COSMOS_DB_CONTAINER"))
print("COSMOS_DB_COUNTERS_CONTAINER:", os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"))
print("COSMOS_DB_LEASE_CONTAINER:", os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"))
print("COSMOS_DB_THROUGHPUT_MODE:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))
print("COSMOS_DB_AUTOSCALE_MAX_RU:", os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "(NA)"))

In [ ]:
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential

# Credential priority: explicit cosmos_credential > explicit cosmos_key > DefaultAzureCredential.
# Set COSMOS_DB_KEY in your .env if you don't yet have control-plane RBAC (currently in private preview).
memory = AsyncCosmosMemoryClient(
    cosmos_endpoint=os.getenv("COSMOS_DB_ENDPOINT"),
    cosmos_database=os.getenv("COSMOS_DB_DATABASE"),
    cosmos_container=os.getenv("COSMOS_DB_CONTAINER"),
    cosmos_counter_container=os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"),
    cosmos_lease_container=os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"),
    cosmos_throughput_mode=os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"),
    cosmos_autoscale_max_ru=int(os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "1000")),
    cosmos_key=os.getenv("COSMOS_DB_KEY"),
    ai_foundry_endpoint=os.getenv("AI_FOUNDRY_ENDPOINT"),
    ai_foundry_api_key=os.getenv("AI_FOUNDRY_API_KEY"),
    embedding_deployment_name=os.getenv("AI_FOUNDRY_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-large"),
    chat_deployment_name=os.getenv("AI_FOUNDRY_CHAT_DEPLOYMENT_NAME", "gpt-4o-mini"),
    use_default_credential=True,
)

print("AsyncCosmosMemoryClient instance created")
print("Throughput mode:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))
print("Local memory store:", memory.local_memory)


## 2. Local Memory Operations

`AsyncCosmosMemoryClient` mirrors the sync API. Local operations (`add_local`, `get_local`, `update_local`, `delete_local`) are synchronous under the hood (in-memory list), but the class is designed for use in async code paths.

> **Note:** In Jupyter you can `await` directly in cells without wrapping in `asyncio.run()`.

### 2a. Add memories with `add_local`

In [ ]:
import uuid
THREAD_ID = str(uuid.uuid4())
# Use a unique user_id per demo run so we get a clean extraction without
# inheriting facts from prior runs that would dedup new content away.
USER_ID = f"user-{uuid.uuid4().hex[:8]}"
print(f"User ID:   {USER_ID}")
print(f"Thread ID: {THREAD_ID}\n")


In [ ]:
# Add sample conversation: weather in Seattle → booking a trip (6 turns)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="What's the weather like in Seattle this weekend?",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="This weekend Seattle will be around 55°F with partly cloudy skies on Saturday and light rain expected Sunday.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="That sounds nice enough. Can you help me book a trip to Seattle for this weekend?",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Sure! I found round-trip flights departing Friday evening and returning Sunday night. There are also several hotels in downtown Seattle with availability. Would you like me to look at specific airlines or neighborhoods?",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="I found a round-trip on Alaska Airlines for $275 and two hotel options within a 5-minute walk of Pike Place Market: the Inn at the Market ($189/night) and a Hilton Garden Inn ($145/night). Want me to reserve one of these?",
)

memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="Whenever you book a flight for me, always book an aisle seat — never a window or middle.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Got it. I'll always select an aisle seat for your bookings.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="For trip planning, my workflow is: first check the weather for the destination, then check flights, then book the hotel last after everything else is confirmed.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Noted — I'll follow that order: weather, then flights, then hotel.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="And never book me into anything that departs or arrives between midnight and 6am unless I explicitly approve it.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Will do — no overnight bookings without your explicit approval.",
)

print(f"Added {len(memory.local_memory)} memories")
print(json.dumps(memory.get_local(), indent=2))

# A second short thread of pure procedural-style instructions. Demonstrates
# that the extractor produces clean procedural items when the conversation is
# focused on rules/workflows rather than mixed with factual booking specifics.
RULES_THREAD_ID = str(uuid.uuid4())
for role, content in [
    ("user",  "Whenever you book a flight for me, always book an aisle seat — never a window or middle."),
    ("agent", "Got it. I'll always select an aisle seat for your bookings."),
    ("user",  "For trip planning, my workflow is: first check the weather, then check flights, and book the hotel last after everything else is confirmed."),
    ("agent", "Noted — I'll follow that order: weather, then flights, then hotel."),
    ("user",  "Never book me into anything that departs or arrives between midnight and 6am unless I explicitly approve it."),
    ("agent", "Will do — no overnight bookings without your explicit approval."),
    ("user",  "When picking a hotel, only recommend ones that include complimentary breakfast."),
    ("agent", "Understood — only hotels with complimentary breakfast."),
]:
    memory.add_local(user_id=USER_ID, role=role, thread_id=RULES_THREAD_ID, content=content)

print(f"Rules thread ID: {RULES_THREAD_ID} ({sum(1 for m in memory.local_memory if m['thread_id']==RULES_THREAD_ID)} turns)")


### 2b. Query memories with `get_local`

In [15]:
# Get all memories
all_memories = memory.get_local()
print(f"Total memories: {len(all_memories)}\n")

# Filter by user_id
user_memories = memory.get_local(user_id=USER_ID)
print(f"Memories for {USER_ID}: {len(user_memories)}")

# Filter by role
agent_memories = memory.get_local(role="agent")
print(f"Agent memories: {len(agent_memories)}")
for m in agent_memories:
    print(f"  [{m['id'][:8]}...] {m['content'][:60]}")

# Filter by type
facts = memory.get_local(memory_types=["fact"])
print(f"\nFact memories: {len(facts)}")
for m in facts:
    print(f"  [{m['id'][:8]}...] {m['content']}")

# Combine filters: current user + agent role
user_agent_memories = memory.get_local(user_id=USER_ID, role="agent")
print(f"\nAgent memories for {USER_ID}: {len(user_agent_memories)}")

Total memories: 19

Memories for user-68421be1: 19
Agent memories: 10
  [4d27c469...] This weekend Seattle will be around 55°F with partly cloudy 
  [5991e394...] Sure! I found round-trip flights departing Friday evening an
  [ae314a6e...] I found a round-trip on Alaska Airlines for $275 and two hot
  [bc5681e4...] Got it. I'll always select an aisle seat for your bookings.
  [eccd2557...] Noted — I'll follow that order: weather, then flights, then 
  [d2109708...] Will do — no overnight bookings without your explicit approv
  [f0fd3216...] Got it. I'll always select an aisle seat for your bookings.
  [d90de470...] Noted — I'll follow that order: weather, then flights, then 
  [5156acad...] Will do — no overnight bookings without your explicit approv
  [5ba8a11c...] Understood — only hotels with complimentary breakfast.

Fact memories: 0

Agent memories for user-68421be1: 10


### 2c. Update & Delete with `update_local` / `delete_local`

In [ ]:
# Update the user's budget constraint to be more specific
target_id = memory.local_memory[4]["id"]  # "Something near Pike Place Market..."
print(f"Before update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}\n")

memory.update_local(
    memory_id=target_id,
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)
print(f"After update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}")

# Delete the third memory (index 2 — the user's booking request)
del_target_id = memory.local_memory[2]["id"]
print(f"\nDeleting memory {del_target_id[:8]}...")
memory.delete_local(del_target_id)

# Verify it's gone
print(f"\nRemaining memories: {len(memory.get_local())}")
for m in memory.get_local():
    print(f" [{m['thread_id'][:8]}...]  [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:50]}")

## 3. Cosmos DB Operations

### 3a. Connect and create the memory store

The async client auto-connects on the first Cosmos DB operation. Call `create_memory_store()` to create the database and container if they do not already exist, including the hierarchical partition key, vector index, and full-text index.

> **Note:** `create_memory_store()` is safe to run more than once.

In [ ]:
# The async client auto-creates the database and container on the first Cosmos operation.
# You can also call create_memory_store() explicitly if needed.
await memory.create_memory_store()
print(f"Connected: {memory._container_client is not None}")

### 3b. Add memories to Cosmos DB with `add_cosmos`

In [ ]:
await memory.connect_cosmos()

In [ ]:
await memory.push_to_cosmos()

# Push a new thread directly to Cosmos DB without adding to local memory first
new_thread_id = str(uuid.uuid4())
print(f"New Thread ID: {new_thread_id}\n")

# Add memories directly to Cosmos DB using add_cosmos
await memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="Can you recommend some good restaurants in New York City?",
)
await memory.add_cosmos(
    user_id="user-002", role="tool", thread_id=new_thread_id,
    content='{"query": "top restaurants NYC", "results": ["Carbone", "Nobu", "Katz\'s Deli", "Le Bernardin"]}',
    metadata={"tool_name": "restaurant_search", "tool_call_id": "call_abc123"},
)
await memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="Absolutely! NYC has incredible dining options. For Italian, try Carbone in Greenwich Village. For sushi, Nobu in Tribeca is world-class. For a classic NYC experience, Katz's Delicatessen on the Lower East Side is a must.",
)
await memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="I love Italian food. Are there any options that are budget-friendly?",
)
await memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="For budget-friendly Italian in NYC, check out L'industrie Pizzeria in Williamsburg or Artichoke Basille's Pizza. Both are highly rated and won't break the bank.",
)

# Verify the memories were added directly to Cosmos DB (not in local memory)
print(f"Local memory count (should be unchanged): {len(memory.local_memory)}\n")

cosmos_results = await memory.get_memories(user_id="user-002", thread_id=new_thread_id)
print(f"Memories in Cosmos DB for new thread: {len(cosmos_results)}")
for r in cosmos_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} {r['content'][:60]}")

### 3c. Retrieve memories from Cosmos DB with `get_memories`

Supports the same filters as `get_local`: `memory_id`, `user_id`, `role`, `memory_type`.

In [ ]:
# Get all memories for the current demo user
results = await memory.get_memories(user_id=USER_ID)
print(f"Memories for {USER_ID}: {len(results)}\n")
for r in results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

# Get only agent memories
agent_results = await memory.get_memories(role="agent")
print(f"\nAgent memories: {len(agent_results)}")
for r in agent_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

### 3d. Update & Delete in Cosmos DB

If the content changes, the embedding is automatically re-generated (awaited).

In [ ]:
# Update the user's budget message to add a hotel budget constraint
user_msgs = await memory.get_memories(user_id=USER_ID, role="user")
target = [m for m in user_msgs if "Pike Place" in m["content"]][0]
print(f"Before: {target['content']}\n")

await memory.update_cosmos(
    memory_id=target["id"],
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)

updated = (await memory.get_memories(memory_id=target["id"]))[0]
print(f"After:  {updated['content']}")

In [ ]:
# Delete the tool memory from Cosmos (async)
tool_mems = await memory.get_memories(user_id="user-002", role="tool")
print(tool_mems[0])
if tool_mems:
    await memory.delete_cosmos(
        tool_mems[0]["id"],
        thread_id=tool_mems[0]["thread_id"],
        user_id=tool_mems[0]["user_id"],
    )
    print(f"Deleted tool memory {tool_mems[0]['id'][:8]}...")

# Verify
remaining = await memory.get_memories()
print(f"\nRemaining memories in Cosmos DB: {len(remaining)}")
for r in remaining:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

### 3e. Retrieve a full thread with `get_thread`

Same parameters: `thread_id` (required), `user_id` (optional), `recent_k` (optional).

In [ ]:
# Use the CURRENT seed thread (from cell 5) so we operate on the data we just wrote.
thread_id = THREAD_ID
print(f"Using thread_id: {thread_id}\n")

# Get all documents in the thread
thread_all = await memory.get_thread(thread_id=thread_id)
print(f"All memories in thread: {len(thread_all)}")
for m in thread_all:
    print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m.get('role','?'):6} type={m['type']:8}  {m['content'][:60]}")

recent = await memory.get_thread(thread_id=thread_id, recent_k=2)
print(f"\nMost recent 2 memories:")
for m in recent:
    print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m.get('role','?'):6} type={m['type']:8}  {m['content'][:60]}")

thread_user = await memory.get_thread(thread_id=thread_id, user_id=USER_ID)
print(f"\nThread memories for {USER_ID}: {len(thread_user)}")


## 4. Thread Summary (in-process)

`AsyncCosmosMemoryClient.generate_thread_summary()` runs the summarisation pipeline **in-process** — no Azure Functions required. The async client uses the AI Foundry async openai client and the async Cosmos SDK, so calls don't block the event loop.

If a prior summary exists, the call performs an **incremental update** that preserves the metadata-tracked `source_count`.


In [ ]:
thread_id = THREAD_ID
user_id = USER_ID
print(f"Summarizing thread_id={thread_id}  user_id={user_id}\n")

summary_doc = await memory.generate_thread_summary(user_id=user_id, thread_id=thread_id)


In [ ]:
print("Summary document:")
print(f"  id: {summary_doc['id']}")
print(f"  content: {summary_doc['content'][:200]}...")
print(f"  source_count: {summary_doc.get('metadata', {}).get('source_count')}")
print(f"  embedding length: {len(summary_doc.get('embedding', []))}")


## 5. Memory Extraction (facts + episodic + procedural)

`extract_memories()` runs a single LLM call that produces three structured memory types:

| Type         | Description                                                              |
|--------------|--------------------------------------------------------------------------|
| `fact`       | Stable, atomic facts about the user.                                     |
| `episodic`   | Discrete past events the user described.                                 |
| `procedural` | How-to guidance the agent should follow.                                 |

Each derived memory is embedded and stored in Cosmos DB with auto-generated tags and a salience score.


In [ ]:
# Extract memories from both the SEED and the pure-procedural RULES thread.
result_main = await memory.extract_memories(user_id=USER_ID, thread_id=THREAD_ID)
print("Seed thread extraction:", result_main)

result_rules = await memory.extract_memories(user_id=USER_ID, thread_id=RULES_THREAD_ID)
print("Rules thread extraction:", result_rules)


In [ ]:
# Inspect the persisted derived memories across BOTH threads for this user.
for mt in ("fact", "episodic", "procedural"):
    docs = await memory.get_memories(user_id=USER_ID, memory_types=[mt])
    print(f"\n{mt.upper()}S ({len(docs)}):")
    for d in docs:
        print(f"  • {d['content'][:100]}  [salience={d.get('salience')}]")


## 6. User Summary (cross-thread profile)

`generate_user_summary()` aggregates memories **across all threads** for a user and produces a structured profile (preferences, account state, behavioural patterns, …). The result is stored in Cosmos DB with `type: "user_summary"`.

Retrieve the latest stored profile at any time with `get_user_summary(user_id)`.


In [ ]:
# Generate a user-level summary across all threads.
user_summary_doc = await memory.generate_user_summary(user_id=user_id)
print("User summary id:", user_summary_doc["id"])
print("\nContent:\n", user_summary_doc["content"][:600], "...")
print("\nStructured profile keys:", list(user_summary_doc.get("metadata", {}).get("structured_summary", {}).keys()))


In [ ]:
# Retrieve the stored user summary from Cosmos DB (async)
stored = await memory.get_user_summary(user_id=user_id)
if stored:
    print("User Summary for", user_id)
    print(stored["content"])
else:
    print("No user summary found — run the generate_user_summary cell first.")


### 7. Vector search with `search_cosmos`

Same as the sync version — embeds the query and runs a `VectorDistance` similarity search.

In [ ]:
results_search_async = await memory.search_cosmos(
    search_terms="What did the user ask about the weather?",
    user_id=USER_ID,
    top_k=3, hybrid_search= True
)

In [ ]:
print(f"Top {len(results_search_async)} results:\n")
for r in results_search_async:
    score = r.get("score", "N/A")
    print(f"  [{r['id'][:8]}...] score={score}  {r['content'][:60]}")

In [ ]:
# Clean up async clients when done
await memory.close()
print("Async clients closed.")